In [0]:
import requests
import pandas as pd
from pyspark.sql import SparkSession

def extrair_e_carregar_previsao_clima():
    """
    EXTRACT & LOAD (ELT): Obtém a previsão do tempo diária para os próximos 14 dias
    em Bombinhas, SC, e guarda os dados brutos na camada Bronze (Delta Lake).
    Como a previsão muda constantemente, esta tabela será sobrescrita a cada execução.
    """
    # Coordenadas geográficas oficiais de Bombinhas, SC
    latitude = -27.1595
    longitude = -48.5137
    
    # Endpoint de Previsão (Forecast) da Open-Meteo
    url = "https://api.open-meteo.com/v1/forecast"
    
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum,rain_sum,wind_speed_10m_max",
        "timezone": "America/Sao_Paulo",
        "forecast_days": 14  # 14 dias de previsão para planeamento de tarifas a curto/médio prazo
    }
    
    print("🔄 [Extract] A aceder à API de Previsão Open-Meteo para os próximos 14 dias...")
    
    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        dados_brutos = response.json()
    except Exception as e:
        raise RuntimeError(f"❌ Falha ao consumir a API de Previsão Open-Meteo: {e}")
        
    daily_data = dados_brutos.get("daily", {})
    
    if not daily_data or "time" not in daily_data:
        raise RuntimeError("❌ Nenhum dado de previsão diária foi retornado pela API.")
        
    # 1. Criar DataFrame do Pandas
    df_pandas = pd.DataFrame(daily_data)
    
    # Renomear as colunas para manter o mesmo padrão da tabela de histórico
    df_pandas.rename(columns={
        "time": "data",
        "temperature_2m_max": "temp_max",
        "temperature_2m_min": "temp_min",
        "precipitation_sum": "precipitacao_total",
        "rain_sum": "chuva_total",
        "wind_speed_10m_max": "vento_max"
    }, inplace=True)
    
    # 2. Converter para Spark DataFrame
    df_spark = spark.createDataFrame(df_pandas)
    
    # 3. Garantir a existência do esquema bronze
    print("📁 A garantir que o esquema 'bronze' existe...")
    spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
    
    # 4. [Load] Gravar na tabela Bronze como Delta Lake (sobrescrevendo com a previsão mais recente)
    print("💾 [Load] A guardar previsão do tempo na tabela bronze.previsao_clima_bombinhas_raw...")
    df_spark.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("bronze.previsao_clima_bombinhas_raw")
        
    print("✅ Ingestão de Previsão do Tempo concluída com sucesso!")

# Executar o pipeline de previsão
extrair_e_carregar_previsao_clima()

In [0]:
%sql
SELECT * FROM bronze.previsao_clima_bombinhas_raw ORDER BY data ASC;